This notebook integrates:
- `plot_maps_3.py` (low-res terrain + landuse from NC)
- `plot_tif_data_hgt_1.py` (hi-res SRTM terrain from TIF+NPY)
- `plot_tif_data_lcz_1.py` (hi-res LCZ from TIF+NPY)

Inputs: `./data/terrain_lcz/`  
Outputs: `./publish/`

Produces 4 Geo-style maps (TIFF, 300 dpi):
- `hongkong_hgt_basic_wrfgrid.tif`
- `hongkong_lcz_basic_wrfgrid.tif`
- `hongkong_hgt_srtm_tif.tif`
- `hongkong_lcz_w2w_tif.tif`


In [1]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import numpy as np

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager
from matplotlib.colors import BoundaryNorm, LightSource, ListedColormap

import cartopy.crs as ccrs
import netCDF4 as nc
import rasterio

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Paths (inputs in ./data/terrain_lcz, outputs in ./publish)
# NOTE: VS Code notebooks sometimes run with CWD = workspace root.
#       The fallback below makes paths work whether CWD is Fig6/ or its parent.
# ----------------------------------------------------------------------------
DATA_DIR = Path("data/terrain_lcz")
OUT_DIR = Path("publish")

if not DATA_DIR.is_dir():
    DATA_DIR = Path("Fig6/data/terrain_lcz")
if not OUT_DIR.is_dir():
    OUT_DIR = Path("Fig6/publish")

OUT_DIR.mkdir(parents=True, exist_ok=True)

NC_BASIC = DATA_DIR / "maps_data_d05_era5_no_lcz_no_srtm_24h.nc"
SRTM_TIF = DATA_DIR / "srtm_59_08.tif"
SRTM_NPY = DATA_DIR / "srtm_59_08_hgt.npy"
LCZ_TIF = DATA_DIR / "hongkong_1.tif"
LCZ_NPY = DATA_DIR / "hongkong_lcz.npy"

_required_inputs = [NC_BASIC, SRTM_TIF, SRTM_NPY, LCZ_TIF, LCZ_NPY]
_missing = [p for p in _required_inputs if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "Missing required input files:\n"
        + "\n".join(f"- {p.as_posix()}" for p in _missing)
        + f"\n\nDetected DATA_DIR={DATA_DIR.resolve().as_posix()}"
        + f"\nDetected OUT_DIR={OUT_DIR.resolve().as_posix()}"
    )

print(f"[Paths] DATA_DIR = {DATA_DIR.resolve().as_posix()}")
print(f"[Paths] OUT_DIR  = {OUT_DIR.resolve().as_posix()}")

# ----------------------------------------------------------------------------
# Plot style
# ----------------------------------------------------------------------------
DPI = 600
FONTSIZE = 16
FIG_W = 6  # inches; for low-res NC maps, figure height is computed from extent
FIGSIZE_ELEV = (10, 8)  # inches; keep consistent across elevation plots

# Unify axis tick spacing across all maps
GRID_LON_STEP = 0.1
GRID_LAT_STEP = 0.1

_avail_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT_FAMILY = "Arial" if "Arial" in _avail_fonts else "DejaVu Sans"

mpl.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": FONTSIZE,
        "axes.labelsize": FONTSIZE,
        "xtick.labelsize": FONTSIZE,
        "ytick.labelsize": FONTSIZE,
        "legend.fontsize": FONTSIZE,
    }
)
print(f"[Font] Using: {FONT_FAMILY}")

# Terrain colormap (same as Fig6 scripts)
_terrain_colors = [
    (0.00, "#FFFFFF"),
    (0.04, "#D8F0C8"),
    (0.12, "#A8D890"),
    (0.25, "#78B850"),
    (0.38, "#C8C060"),
    (0.52, "#C8A050"),
    (0.65, "#A07038"),
    (0.80, "#784820"),
    (1.00, "#4A2810"),
]
CMAP_TERRAIN = mcolors.LinearSegmentedColormap.from_list(
    "hk_terrain",
    [(pos, mcolors.to_rgb(col)) for pos, col in _terrain_colors],
    N=512,
)


def save_tiff(fig: "plt.Figure", out_path: Path) -> None:
    """Save a TIFF at 300 dpi; tries LZW compression if supported."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    common = dict(dpi=DPI, format="tiff", bbox_inches="tight")
    try:
        fig.savefig(out_path, pil_kwargs={"compression": "tiff_lzw"}, **common)
    except TypeError:
        fig.savefig(out_path, **common)


[Paths] DATA_DIR = E:/BaiduSyncdisk/Code/06_AI_WRF_UCM/Figs/Fig6/data/terrain_lcz
[Paths] OUT_DIR  = E:/BaiduSyncdisk/Code/06_AI_WRF_UCM/Figs/Fig6/publish
[Font] Using: Arial


In [2]:
# ----------------------------------------------------------------------------
# Category definitions
# ----------------------------------------------------------------------------

# 31 classes (1-21 MODIS + 31-40 urban LCZ extension)
LU_ALL = {
    1: ("#05450a", "Evergreen Needleleaf Forest"),
    2: ("#086a10", "Evergreen Broadleaf Forest"),
    3: ("#54a708", "Deciduous Needleleaf Forest"),
    4: ("#78d203", "Deciduous Broadleaf Forest"),
    5: ("#009900", "Mixed Forests"),
    6: ("#c6b044", "Closed Shrublands"),
    7: ("#dcd159", "Open Shrublands"),
    8: ("#dade48", "Woody Savannas"),
    9: ("#fbff13", "Savannas"),
    10: ("#b6ff05", "Grasslands"),
    11: ("#27ff87", "Permanent Wetlands"),
    12: ("#c24f44", "Croplands"),
    13: ("#a5a5a5", "Urban and Built-Up"),
    14: ("#ff6d4c", "Cropland/Natural Veg. Mosaic"),
    15: ("#69fff8", "Snow and Ice"),
    16: ("#f9ffa4", "Barren or Sparsely Vegetated"),
    17: ("#1c0dff", "Water"),
    18: ("#6e8b3d", "Wooded Tundra"),
    19: ("#9acd32", "Mixed Tundra"),
    20: ("#d2b48c", "Barren Tundra"),
    21: ("#4169e1", "Lakes"),
    31: ("#910613", "Compact High-Rise"),
    32: ("#D9081C", "Compact Mid-Rise"),
    33: ("#FF0A22", "Compact Low-Rise"),
    34: ("#C54F1E", "Open High-Rise"),
    35: ("#FF6628", "Open Mid-Rise"),
    36: ("#FF985E", "Open Low-Rise"),
    37: ("#FDED3F", "Lightweight Low-Rise"),
    38: ("#BBBBBB", "Large Low-Rise"),
    39: ("#FFCBAB", "Sparsely Built"),
    40: ("#565656", "Heavy Industry"),
}

# 21 basic classes (1-21 only)
LU_BASIC = {k: v for k, v in LU_ALL.items() if k <= 21}

# WUDAPT standard (17 categories: 1–10 + A–G mapped to 11–17)
LCZ_INFO = {
    1: ("#910613", "LCZ 1  Compact highrise"),
    2: ("#D9081C", "LCZ 2  Compact midrise"),
    3: ("#FF0A22", "LCZ 3  Compact lowrise"),
    4: ("#C54F1E", "LCZ 4  Open highrise"),
    5: ("#FF6628", "LCZ 5  Open midrise"),
    6: ("#FF985E", "LCZ 6  Open lowrise"),
    7: ("#FDED3F", "LCZ 7  Lightweight low-rise"),
    8: ("#BBBBBB", "LCZ 8  Large lowrise"),
    9: ("#FFCBAB", "LCZ 9  Sparsely built"),
    10: ("#565656", "LCZ 10 Heavy industry"),
    11: ("#006A18", "LCZ A  Dense trees"),
    12: ("#00A926", "LCZ B  Scattered trees"),
    13: ("#628432", "LCZ C  Bush, scrub"),
    14: ("#B5DA7F", "LCZ D  Low plants"),
    15: ("#000000", "LCZ E  Bare rock or paved"),
    16: ("#FCF7B1", "LCZ F  Bare soil or sand"),
    17: ("#656BFA", "LCZ G  Water"),
}


In [3]:
# ----------------------------------------------------------------------------
# Utilities (ported from Fig6 scripts)
# ----------------------------------------------------------------------------

def read_nc(filepath: Path, varnames: list[str]) -> dict[str, np.ndarray]:
    data: dict[str, np.ndarray] = {}
    with nc.Dataset(filepath, "r") as ds:
        for v in varnames:
            data[v] = ds.variables[v][:]
    return data


def get_extent(lon: np.ndarray, lat: np.ndarray) -> list[float]:
    return [float(np.min(lon)), float(np.max(lon)), float(np.min(lat)), float(np.max(lat))]


def calc_figsize(extent: list[float], fig_w: float = FIG_W) -> tuple[float, float]:
    lon_min, lon_max, lat_min, lat_max = extent
    lat_mid = (lat_min + lat_max) / 2.0
    lon_span = (lon_max - lon_min) * math.cos(math.radians(lat_mid))
    lat_span = lat_max - lat_min
    fig_h = round(fig_w / (lon_span / lat_span), 3)
    return fig_w, fig_h


def make_gridlines(
    ax,
    extent: list[float],
    *,
    lon_step: float | None = None,
    lat_step: float | None = None,
    font_family: str = FONT_FAMILY,
    fontsize: int = FONTSIZE,
):
    """Gray dashed gridlines; default adaptive step unless lon_step/lat_step are given."""
    lon_min, lon_max, lat_min, lat_max = extent

    def _nice_step(span: float, target: float = 5) -> float:
        raw = span / target
        mag = 10 ** math.floor(math.log10(raw))
        for s in [1, 2, 2.5, 5, 10]:
            if s * mag >= raw:
                return round(s * mag, 6)
        return round(10 * mag, 6)

    if lon_step is None:
        lon_step = _nice_step(lon_max - lon_min)
    if lat_step is None:
        lat_step = _nice_step(lat_max - lat_min)

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="gray",
        alpha=0.5,
        linestyle="--",
        crs=ccrs.PlateCarree(),
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlocator = mticker.MultipleLocator(lon_step)
    gl.ylocator = mticker.MultipleLocator(lat_step)
    gl.xlabel_style = {"size": fontsize, "family": font_family}
    gl.ylabel_style = {"size": fontsize, "family": font_family}
    return gl


def add_aligned_colorbar(
    fig: "plt.Figure",
    ax,
    mappable,
    *,
    label: str | None = None,
    fontsize: int = FONTSIZE,
    font_family: str = FONT_FAMILY,
    tick_labels: list[str] | None = None,
    tick_vals: list[float] | list[int] | np.ndarray | None = None,
    cbar_width: float = 0.020,
    cbar_pad: float = 0.015,
):
    """Colorbar whose height aligns with the map axes (ported from plot_maps_3.py)."""
    fig.canvas.draw()
    ax_pos = ax.get_position()
    cbar_ax = fig.add_axes([ax_pos.x1 + cbar_pad, ax_pos.y0, cbar_width, ax_pos.height])
    cbar = fig.colorbar(mappable, cax=cbar_ax, orientation="vertical")

    if label:
        cbar.set_label(label, fontsize=fontsize, family=font_family)

    if tick_vals is not None:
        cbar.set_ticks(tick_vals)
    if tick_labels is not None:
        cbar.set_ticklabels(tick_labels, fontsize=fontsize - 1)

    cbar.ax.tick_params(labelsize=fontsize)
    return cbar


In [4]:
# ----------------------------------------------------------------------------
# Low-res (NC/WRF-grid) plots: terrain + LU (basic)
# Ported from plot_maps_3.py
# ----------------------------------------------------------------------------


def plot_hgt_wrfgrid(
    lon: np.ndarray,
    lat: np.ndarray,
    hgt: np.ndarray,
    extent: list[float],
    out_name: str,
    *,
    lu: np.ndarray | None = None,
    water_lu_ids: tuple[int, ...] = (17, 21),
) -> Path:
    print(f"[Plot] {out_name} ...")

    figsize = FIGSIZE_ELEV
    proj = ccrs.PlateCarree()

    hgt_f = np.array(hgt, dtype=np.float32)
    hgt_f = np.where(np.isfinite(hgt_f), hgt_f, np.nan)

    # Mask sea pixels (WRF water category) so ocean area stays blank/white.
    if lu is not None:
        lu_arr = np.array(lu)
        if lu_arr.shape != hgt_f.shape:
            lu_arr = np.squeeze(lu_arr)
        if lu_arr.shape == hgt_f.shape:
            water_mask = np.isin(lu_arr, water_lu_ids)
            hgt_f = np.where(water_mask, np.nan, hgt_f)

    if not np.isfinite(hgt_f).any():
        raise ValueError(f"{out_name}: no finite terrain values after masking")

    vmin_data = float(np.nanmin(hgt_f))
    vmax_data = float(np.nanmax(hgt_f))
    vmax_data = max(vmax_data, vmin_data + 1.0)

    tick_step = max(1, round((vmax_data - vmin_data) / 6 / 50) * 50)
    tick_step = 50 if tick_step == 0 else tick_step

    # Round to "nice" bounds so ticks don't push the colorbar limits and create
    # white margins above/below the filled range.
    vmin = math.floor(vmin_data / tick_step) * tick_step
    vmax = math.ceil(vmax_data / tick_step) * tick_step
    if vmax <= vmin:
        vmax = vmin + tick_step

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    lon_arr = np.array(lon)
    lat_arr = np.array(lat)
    lat_mid = float((np.min(lat_arr) + np.max(lat_arr)) / 2.0)

    if lon_arr.ndim == 2:
        dx_deg = float(np.abs(np.diff(lon_arr, axis=1)).mean())
        dy_deg = float(np.abs(np.diff(lat_arr, axis=0)).mean())
    else:
        dx_deg = float(np.abs(np.diff(lon_arr)).mean()) if lon_arr.size > 1 else 0.01
        dy_deg = float(np.abs(np.diff(lat_arr)).mean()) if lat_arr.size > 1 else 0.01

    dx_m = dx_deg * 111320 * math.cos(math.radians(lat_mid))
    dy_m = dy_deg * 110570

    ls = LightSource(azdeg=315, altdeg=45)
    hgt_for_shade = np.where(np.isfinite(hgt_f), hgt_f, 0.0)
    hillshade = ls.hillshade(hgt_for_shade, vert_exag=3, dx=dx_m, dy=dy_m)
    hillshade = np.where(np.isfinite(hgt_f), hillshade, np.nan)

    rgb_terrain = CMAP_TERRAIN(norm(hgt_f))
    rgb_blend = ls.blend_hsv(
        rgb_terrain[:, :, :3],
        hillshade[:, :, np.newaxis],
        hsv_max_sat=0.7,
        hsv_max_val=0.9,
    )
    alpha_ch = np.where(np.isfinite(hgt_f), 1.0, 0.0)
    rgba_out = np.dstack([rgb_blend, alpha_ch])

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(1, 1, 1, projection=proj)
    ax.set_extent(extent, crs=proj)
    ax.set_facecolor("white")

    ax.pcolormesh(
        lon,
        lat,
        hgt_f,
        cmap=CMAP_TERRAIN,
        norm=norm,
        transform=proj,
        shading="auto",
        alpha=0,  # invisible; only to ensure correct mappable semantics
    )

    # For imshow(), the origin must match the latitude direction in the array.
    # If LAT increases with row index (south->north), use origin='lower'.
    lat_s = np.squeeze(lat_arr)
    if lat_s.ndim in (1, 2):
        origin = "lower" if float(np.nanmean(lat_s[-1])) > float(np.nanmean(lat_s[0])) else "upper"
    else:
        origin = "upper"

    ax.imshow(
        rgba_out,
        origin=origin,
        extent=extent,
        transform=proj,
        interpolation="bilinear",
        zorder=1,
    )

    make_gridlines(ax, extent, lon_step=GRID_LON_STEP, lat_step=GRID_LAT_STEP)

    sm = plt.cm.ScalarMappable(cmap=CMAP_TERRAIN, norm=norm)
    sm.set_array(np.array([vmin, vmax], dtype=np.float32))

    cbar_ticks = np.arange(vmin, vmax + 0.1 * tick_step, tick_step)

    add_aligned_colorbar(
        fig,
        ax,
        sm,
        label="Elevation (m)",
        tick_vals=cbar_ticks,
        tick_labels=[f"{int(t)}" for t in cbar_ticks],
        fontsize=FONTSIZE,
    )

    out_path = OUT_DIR / f"{out_name}.tif"
    save_tiff(fig, out_path)
    plt.close(fig)

    print(
        f"  Saved: {out_path.as_posix()}  "
        f"(figsize={figsize[0]:.2f}×{figsize[1]:.2f} in, elev {vmin_data:.0f}~{vmax_data:.0f} m)"
    )
    return out_path



def plot_lu_wrfgrid(
    lon: np.ndarray,
    lat: np.ndarray,
    lu: np.ndarray,
    lu_dict: dict[int, tuple[str, str]],
    extent: list[float],
    out_name: str,
) -> Path:
    print(f"[Plot] {out_name} ...")

    lu_arr = np.array(lu)

    present_ids = sorted([i for i in lu_dict.keys() if i in np.unique(lu_arr)])
    if not present_ids:
        raise ValueError(f"{out_name}: no valid LU categories found in data")

    colors = [lu_dict[i][0] for i in present_ids]
    labels = [lu_dict[i][1] for i in present_ids]

    boundaries = [present_ids[0] - 0.5]
    for i in range(len(present_ids) - 1):
        boundaries.append((present_ids[i] + present_ids[i + 1]) / 2.0)
    boundaries.append(present_ids[-1] + 0.5)

    cmap = ListedColormap(colors)
    norm = BoundaryNorm(boundaries, ncolors=len(present_ids))

    figsize = calc_figsize(extent)
    proj = ccrs.PlateCarree()

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(1, 1, 1, projection=proj)
    ax.set_extent(extent, crs=proj)
    ax.set_facecolor("white")

    im = ax.pcolormesh(
        lon,
        lat,
        lu_arr,
        cmap=cmap,
        norm=norm,
        transform=proj,
        shading="auto",
    )

    make_gridlines(ax, extent, lon_step=GRID_LON_STEP, lat_step=GRID_LAT_STEP)

    add_aligned_colorbar(
        fig,
        ax,
        im,
        tick_vals=present_ids,
        tick_labels=labels,
        fontsize=FONTSIZE,
    )

    out_path = OUT_DIR / f"{out_name}.tif"
    save_tiff(fig, out_path)
    plt.close(fig)

    print(f"  Saved: {out_path.as_posix()}  ({len(present_ids)} categories)")
    return out_path


In [5]:
# Generate low-res (WRF-grid) maps from NC (basic dataset)

VARS = ["LON", "LAT", "HGT", "LU_INDEX"]

d_basic = read_nc(NC_BASIC, VARS)
extent_basic = get_extent(d_basic["LON"], d_basic["LAT"])
print(f"[NC basic extent] {[round(v, 4) for v in extent_basic]}")

out_hgt_basic = plot_hgt_wrfgrid(
    d_basic["LON"],
    d_basic["LAT"],
    d_basic["HGT"],
    extent_basic,
    out_name="hongkong_hgt_basic_wrfgrid",
    lu=d_basic["LU_INDEX"],
)

out_lu_basic = plot_lu_wrfgrid(
    d_basic["LON"],
    d_basic["LAT"],
    d_basic["LU_INDEX"],
    LU_BASIC,
    extent_basic,
    out_name="hongkong_lcz_basic_wrfgrid",
)

(out_hgt_basic, out_lu_basic)


[NC basic extent] [113.8257, 114.4156, 22.1405, 22.5601]
[Plot] hongkong_hgt_basic_wrfgrid ...
  Saved: publish/hongkong_hgt_basic_wrfgrid.tif  (figsize=10.00×8.00 in, elev -0~751 m)
[Plot] hongkong_lcz_basic_wrfgrid ...
  Saved: publish/hongkong_lcz_basic_wrfgrid.tif  (15 categories)


(WindowsPath('publish/hongkong_hgt_basic_wrfgrid.tif'),
 WindowsPath('publish/hongkong_lcz_basic_wrfgrid.tif'))

In [6]:
# ----------------------------------------------------------------------------
# Hi-res terrain (SRTM) from TIF+NPY
# Ported from plot_tif_data_hgt_1.py
# ----------------------------------------------------------------------------

ROI_LON_MIN, ROI_LON_MAX = 113.8257, 114.4156
ROI_LAT_MIN, ROI_LAT_MAX = 22.1405, 22.5601


def plot_hgt_srtm_tif(
    tif_path: Path,
    npy_path: Path,
    *,
    out_name: str = "hongkong_hgt_srtm_tif",
    lon_min: float = ROI_LON_MIN,
    lon_max: float = ROI_LON_MAX,
    lat_min: float = ROI_LAT_MIN,
    lat_max: float = ROI_LAT_MAX,
) -> Path:
    print(f"[Plot] {out_name} ...")

    with rasterio.open(tif_path) as src:
        transform = src.transform
        nodata = src.nodata if src.nodata is not None else -32768.0

    data_full = np.load(npy_path).astype(np.float32)
    rows_total, cols_total = data_full.shape

    x0 = transform.c
    y0 = transform.f
    dx = transform.a
    dy = transform.e

    col_min = max(0, int(np.floor((lon_min - x0) / dx)))
    col_max = min(cols_total, int(np.ceil((lon_max - x0) / dx)))
    row_min = max(0, int(np.floor((y0 - lat_max) / abs(dy))))
    row_max = min(rows_total, int(np.ceil((y0 - lat_min) / abs(dy))))

    data_crop = data_full[row_min:row_max, col_min:col_max].copy()

    actual_lon_min = x0 + col_min * dx
    actual_lon_max = x0 + col_max * dx
    actual_lat_max = y0 + row_min * dy
    actual_lat_min = y0 + row_max * dy

    extent = [actual_lon_min, actual_lon_max, actual_lat_min, actual_lat_max]
    print(
        "[SRTM extent aligned] "
        f"lon {actual_lon_min:.6f}~{actual_lon_max:.6f}, "
        f"lat {actual_lat_min:.6f}~{actual_lat_max:.6f}, "
        f"shape {data_crop.shape}"
    )

    valid_mask = data_crop != nodata
    data_masked = np.where(valid_mask, data_crop, np.nan)

    vmin_data = float(np.nanmin(data_masked))
    vmax_data = float(np.nanmax(data_masked))
    vmax_data = max(vmax_data, vmin_data + 1.0)

    tick_step = max(1, round((vmax_data - vmin_data) / 6 / 50) * 50)
    tick_step = 50 if tick_step == 0 else tick_step

    vmin = math.floor(vmin_data / tick_step) * tick_step
    vmax = math.ceil(vmax_data / tick_step) * tick_step
    if vmax <= vmin:
        vmax = vmin + tick_step

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    ls = LightSource(azdeg=315, altdeg=45)
    dx_m = abs(dx) * 111320 * np.cos(np.radians((actual_lat_min + actual_lat_max) / 2))
    dy_m = abs(dy) * 110570

    data_for_shade = np.where(valid_mask, data_crop, 0.0)
    hillshade = ls.hillshade(data_for_shade, vert_exag=3, dx=dx_m, dy=dy_m)
    hillshade = np.where(valid_mask, hillshade, np.nan)

    rgb_terrain = CMAP_TERRAIN(norm(data_masked))
    rgb_blend = ls.blend_hsv(
        rgb_terrain[:, :, :3],
        hillshade[:, :, np.newaxis],
        hsv_max_sat=0.7,
        hsv_max_val=0.9,
    )
    alpha_channel = np.where(valid_mask, 1.0, 0.0)
    rgba_final = np.dstack([rgb_blend, alpha_channel])

    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=FIGSIZE_ELEV)
    ax = fig.add_subplot(1, 1, 1, projection=proj)
    ax.set_extent(extent, crs=proj)
    ax.set_facecolor("white")

    ax.imshow(
        rgba_final,
        origin="upper",
        extent=extent,
        transform=proj,
        interpolation="bilinear",
        zorder=1,
    )

    make_gridlines(ax, extent, lon_step=GRID_LON_STEP, lat_step=GRID_LAT_STEP)

    sm = plt.cm.ScalarMappable(cmap=CMAP_TERRAIN, norm=norm)
    sm.set_array(np.array([vmin, vmax], dtype=np.float32))

    cbar_ticks = np.arange(vmin, vmax + 0.1 * tick_step, tick_step)

    add_aligned_colorbar(
        fig,
        ax,
        sm,
        label="Elevation (m)",
        tick_vals=cbar_ticks,
        tick_labels=[f"{int(t)}" for t in cbar_ticks],
        fontsize=FONTSIZE,
    )

    out_path = OUT_DIR / f"{out_name}.tif"
    save_tiff(fig, out_path)
    plt.close(fig)

    print(f"  Saved: {out_path.as_posix()}  (elev {vmin_data:.1f}~{vmax_data:.1f} m)")
    return out_path


out_hgt_srtm = plot_hgt_srtm_tif(SRTM_TIF, SRTM_NPY)
out_hgt_srtm


[Plot] hongkong_hgt_srtm_tif ...
[SRTM extent aligned] lon 113.825000~114.415833, lat 22.140000~22.560833, shape (505, 709)
  Saved: publish/hongkong_hgt_srtm_tif.tif  (elev -50.0~943.0 m)


WindowsPath('publish/hongkong_hgt_srtm_tif.tif')

In [7]:
# ----------------------------------------------------------------------------
# Hi-res LCZ (WUDAPT) from TIF+NPY
# Ported from plot_tif_data_lcz_1.py
# ----------------------------------------------------------------------------


def plot_lcz_w2w_tif(
    tif_path: Path,
    npy_path: Path,
    *,
    out_name: str = "hongkong_lcz_w2w_tif",
    lon_min: float = ROI_LON_MIN,
    lon_max: float = ROI_LON_MAX,
    lat_min: float = ROI_LAT_MIN,
    lat_max: float = ROI_LAT_MAX,
) -> Path:
    print(f"[Plot] {out_name} ...")

    with rasterio.open(tif_path) as src:
        transform = src.transform
        nodata = src.nodata if src.nodata is not None else 255

    data_full = np.load(npy_path)
    rows_total, cols_total = data_full.shape

    x0 = transform.c
    y0 = transform.f
    dx = transform.a
    dy = transform.e

    col_min = max(0, int(np.floor((lon_min - x0) / dx)))
    col_max = min(cols_total, int(np.ceil((lon_max - x0) / dx)))
    row_min = max(0, int(np.floor((y0 - lat_max) / abs(dy))))
    row_max = min(rows_total, int(np.ceil((y0 - lat_min) / abs(dy))))

    data_crop = data_full[row_min:row_max, col_min:col_max]

    actual_lon_min = x0 + col_min * dx
    actual_lon_max = x0 + col_max * dx
    actual_lat_max = y0 + row_min * dy
    actual_lat_min = y0 + row_max * dy

    extent = [actual_lon_min, actual_lon_max, actual_lat_min, actual_lat_max]
    print(
        "[LCZ extent aligned] "
        f"lon {actual_lon_min:.6f}~{actual_lon_max:.6f}, "
        f"lat {actual_lat_min:.6f}~{actual_lat_max:.6f}, "
        f"shape {data_crop.shape}"
    )

    valid_mask = data_crop != nodata
    present_vals = sorted([v for v in np.unique(data_crop[valid_mask]) if int(v) in LCZ_INFO])
    present_vals = [int(v) for v in present_vals]

    if not present_vals:
        raise ValueError("No valid LCZ categories found after masking nodata")

    colors_list = [LCZ_INFO[v][0] for v in present_vals]
    cmap = mcolors.ListedColormap(colors_list)
    bounds = [v - 0.5 for v in present_vals] + [present_vals[-1] + 0.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    data_masked = np.ma.masked_where(data_crop == nodata, data_crop)

    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=FIGSIZE_ELEV)
    ax = fig.add_subplot(1, 1, 1, projection=proj)
    ax.set_extent(extent, crs=proj)
    ax.set_facecolor("white")

    im = ax.imshow(
        data_masked,
        origin="upper",
        extent=extent,
        cmap=cmap,
        norm=norm,
        transform=proj,
        interpolation="nearest",
        zorder=1,
    )

    make_gridlines(ax, extent, lon_step=GRID_LON_STEP, lat_step=GRID_LAT_STEP)

    add_aligned_colorbar(
        fig,
        ax,
        im,
        tick_vals=present_vals,
        tick_labels=[LCZ_INFO[v][1] for v in present_vals],
        fontsize=FONTSIZE,
    )

    out_path = OUT_DIR / f"{out_name}.tif"
    save_tiff(fig, out_path)
    plt.close(fig)

    print(f"  Saved: {out_path.as_posix()}  ({len(present_vals)} categories)")
    return out_path


out_lcz_w2w = plot_lcz_w2w_tif(LCZ_TIF, LCZ_NPY)
out_lcz_w2w


[Plot] hongkong_lcz_w2w_tif ...
[LCZ extent aligned] lon 113.825530~114.415723, lat 22.139878~22.560290, shape (468, 657)
  Saved: publish/hongkong_lcz_w2w_tif.tif  (16 categories)


WindowsPath('publish/hongkong_lcz_w2w_tif.tif')

In [8]:
# ----------------------------------------------------------------------------
# Output check
# ----------------------------------------------------------------------------

outputs = [
    OUT_DIR / "hongkong_hgt_basic_wrfgrid.tif",
    OUT_DIR / "hongkong_lcz_basic_wrfgrid.tif",
    OUT_DIR / "hongkong_hgt_srtm_tif.tif",
    OUT_DIR / "hongkong_lcz_w2w_tif.tif",
]

for p in outputs:
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"OK  {p.as_posix()}  ({size_mb:.2f} MB)")
    else:
        print(f"MISS {p.as_posix()}")


OK  publish/hongkong_hgt_basic_wrfgrid.tif  (20.60 MB)
OK  publish/hongkong_lcz_basic_wrfgrid.tif  (2.09 MB)
OK  publish/hongkong_hgt_srtm_tif.tif  (22.41 MB)
OK  publish/hongkong_lcz_w2w_tif.tif  (3.64 MB)
